## 1. Load your recording


In [ ]:
import io
import json
import zipfile
import numpy as np
import ipywidgets as widgets
from IPython.display import display

heading = widgets.HTML("""
<div style="margin-bottom:12px">
    <h2 style="margin:0 0 8px">Start here · Load your recording</h2>
    <p style="margin:0; font-size:15px">
        Select the <b>ZIP recording exported from the virtual rig</b>.
        Your voltage data and stimulus information will be loaded automatically.
        No need to unzip the file yourself.
    </p>
</div>
""")

uploader = widgets.FileUpload(
    accept='.zip',
    multiple=False,
    description='Upload ZIP data',
    icon='upload',
    button_style='primary',
    layout=widgets.Layout(width='260px', height='48px')
)

status = widgets.HTML()
details = widgets.Output()

metadata = stimuli = voltage = fs = filename = None

def load_recording(change):
    global metadata, stimuli, voltage, fs, filename

    if not uploader.value:
        return

    metadata = stimuli = voltage = fs = filename = None
    details.clear_output()
    status.value = '<p style="color:#555">Reading and unpacking recording…</p>'

    try:
        # Compatible with ipywidgets 7 and 8
        if isinstance(uploader.value, dict):
            selected_name, item = next(iter(uploader.value.items()))
        else:
            item = uploader.value[0]
            selected_name = item['name']

        with zipfile.ZipFile(io.BytesIO(bytes(item['content']))) as z:
            new_metadata = json.loads(z.read('metadata.json'))
            new_stimuli = json.loads(z.read('stimuli.json'))
            new_voltage = np.frombuffer(
                z.read('voltage.bin'), dtype='<f4'
            ).copy()

        new_fs = float(new_metadata['sampling_rate_hz'])
        if not np.isfinite(new_fs) or new_fs <= 0:
            raise ValueError('The recording has an invalid sampling rate.')

        metadata, stimuli = new_metadata, new_stimuli
        voltage, fs, filename = new_voltage, new_fs, selected_name

        status.value = """
        <p style="color:#237744; font-size:15px">
            <b>✓ Recording loaded. Continue to the next cell.</b>
        </p>
        """

        with details:
            print(f'File:          {filename}')
            print(f'Duration:      {len(voltage) / fs:.1f} seconds')
            print(f'Sampling rate: {fs:,.0f} Hz')
            print(f'Samples:       {len(voltage):,}')

    except Exception as error:
        status.value = """
        <p style="color:#b42318">
            <b>Could not load the recording.</b>
            Select a ZIP exported from the virtual rig.
        </p>
        """
        with details:
            print(str(error))

uploader.observe(load_recording, names='value')

display(widgets.VBox(
    [heading, uploader, status, details],
    layout=widgets.Layout(
        padding='20px',
        border='1px solid #dce3eb',
        max_width='750px'
    )
))

## 2. Inspect the spatial-mapping recording

The upper panel shows the filtered voltage and automatic detection threshold.
The lower panel marks detected events with thin vertical ticks.

Coloured backgrounds indicate stimulus presentations at different screen
locations. Both panels share the same time axis.

**Explore:** Zoom into individual presentations. Can you identify locations
that produce more activity than others?

In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.signal import butter, sosfiltfilt, find_peaks

# ============================================================
# FILTER AND DETECT EVENTS
# ============================================================

threshold_factor = 4.5
min_separation_ms = 1.0
monitor_y_increases_downward = True

sos = butter(3, 300, btype='highpass', fs=fs, output='sos')
voltage_filtered = sosfiltfilt(sos, voltage)
t = np.arange(len(voltage_filtered)) / fs

signal_median = np.median(voltage_filtered)
noise_sigma = np.median(
    np.abs(voltage_filtered - signal_median)
) / 0.67449

threshold = signal_median - threshold_factor * noise_sigma

spike_indices, _ = find_peaks(
    -voltage_filtered,
    height=-threshold,
    distance=max(1, round(min_separation_ms / 1000 * fs))
)

spike_times = spike_indices / fs

# ============================================================
# LOCATION COLOURS
# ============================================================

# Select scripted trials and round coordinates for consistent grouping.
# Work on copies so the original stimulus log stays unchanged.
spatial_trials = []

for s in stimuli:
    p = s['parameters']

    if (
        s.get('source') == 'protocol'
        and p['type'] == 'grating'
        and 'x' in p
        and 'y' in p
    ):
        trial = {
            **s,
            'parameters': {
                **p,
                'x': round(float(p['x']), 6),
                'y': round(float(p['y']), 6)
            }
        }
        spatial_trials.append(trial)

if not spatial_trials:
    raise ValueError('No scripted grating presentations with x/y positions were found.')

locations = sorted({
    (float(s['parameters']['x']), float(s['parameters']['y']))
    for s in spatial_trials
}, key=lambda p: (p[1], p[0]))

x_unique = np.sort(np.unique([p[0] for p in locations]))
y_unique = np.sort(np.unique([p[1] for p in locations]))

corner_00 = np.array([65, 110, 220])
corner_10 = np.array([220, 75, 125])
corner_01 = np.array([40, 175, 140])
corner_11 = np.array([230, 180, 55])

location_colors = {}
location_ids = np.full((len(y_unique), len(x_unique)), np.nan)
discrete_colorscale = []

for i, (x, y) in enumerate(locations):
    u = (
        (x - x_unique.min()) / np.ptp(x_unique)
        if len(x_unique) > 1 else 0.5
    )
    v = (
        (y - y_unique.min()) / np.ptp(y_unique)
        if len(y_unique) > 1 else 0.5
    )

    rgb = (
        (1 - u) * (1 - v) * corner_00
        + u * (1 - v) * corner_10
        + (1 - u) * v * corner_01
        + u * v * corner_11
    ).round().astype(int)

    color = f'rgb({rgb[0]},{rgb[1]},{rgb[2]})'
    location_colors[(x, y)] = color

    xi = np.searchsorted(x_unique, x)
    yi = np.searchsorted(y_unique, y)
    location_ids[yi, xi] = i

    discrete_colorscale.extend([
        [i / len(locations), color],
        [(i + 1) / len(locations), color]
    ])

# ============================================================
# COMPACT, RESPONSIVE FIGURE
# ============================================================

fig_spatial = make_subplots(
    rows=2,
    cols=2,
    specs=[
        [{'type': 'xy'}, {'type': 'xy', 'rowspan': 2}],
        [{'type': 'xy'}, None]
    ],
    shared_xaxes=True,
    column_widths=[0.76, 0.24],
    row_heights=[0.78, 0.22],
    horizontal_spacing=0.06,
    vertical_spacing=0.08
)

fig_spatial.add_trace(go.Scattergl(
    x=t,
    y=voltage_filtered,
    mode='lines',
    line=dict(color='black', width=0.7),
    showlegend=False,
    hovertemplate=(
        'Time: %{x:.4f} s'
        '<br>Voltage: %{y:.1f} µV<extra></extra>'
    )
), row=1, col=1)

fig_spatial.add_trace(go.Scattergl(
    x=spike_times,
    y=np.zeros(len(spike_times)),
    mode='markers',
    marker=dict(
        symbol='line-ns',
        size=12,
        color='black',
        line=dict(color='black', width=0.5)
    ),
    showlegend=False,
    hovertemplate='Detected event: %{x:.4f} s<extra></extra>'
), row=2, col=1)

shapes = []

for s in spatial_trials:
    position = (
        float(s['parameters']['x']),
        float(s['parameters']['y'])
    )

    for plot_row in [1, 2]:
        subplot = fig_spatial.get_subplot(plot_row, 1)
        xref = subplot.xaxis.plotly_name.replace('axis', '')
        yref = subplot.yaxis.plotly_name.replace('axis', '')

        shapes.append(dict(
            type='rect',
            xref=xref,
            yref=f'{yref} domain',
            x0=s['onset_s'],
            x1=s['offset_s'],
            y0=0, y1=1,
            fillcolor=location_colors[position],
            opacity=0.22,
            line=dict(width=0),
            layer='below'
        ))

fig_spatial.update_layout(shapes=shapes)

fig_spatial.add_hline(
    y=threshold,
    row=1, col=1,
    line_color='#d62728',
    line_dash='dash',
    line_width=1.5,
    annotation_text=f'Threshold: {threshold:.1f} µV',
    annotation_position='top right'
)

# ============================================================
# MONITOR KEY
# ============================================================

fig_spatial.add_trace(go.Heatmap(
    x=[f'{x:g}' for x in x_unique],
    y=[f'{y:g}' for y in y_unique],
    z=location_ids,
    zmin=-0.5,
    zmax=len(locations) - 0.5,
    colorscale=discrete_colorscale,
    showscale=False,
    zsmooth=False,
    xgap=1,
    ygap=1,
    hoverongaps=False,
    hovertemplate='X: %{x}<br>Y: %{y}<extra></extra>'
), row=1, col=2)

monitor_subplot = fig_spatial.get_subplot(1, 2)
monitor_xref = monitor_subplot.xaxis.plotly_name.replace('axis', '')
monitor_yref = monitor_subplot.yaxis.plotly_name.replace('axis', '')

fig_spatial.update_xaxes(
    title=dict(text='X', standoff=4),
    type='category',
    categoryorder='array',
    categoryarray=[f'{x:g}' for x in x_unique],
    range=[-0.5, len(x_unique) - 0.5],
    showticklabels=False,
    ticks='',
    showgrid=False,
    zeroline=False,
    fixedrange=True,
    constrain='domain',
    row=1, col=2
)

# Maintain the monitor's 16:9 shape as the figure resizes
monitor_y_range = [-0.5, len(y_unique) - 0.5]
if monitor_y_increases_downward:
    monitor_y_range = monitor_y_range[::-1]

fig_spatial.update_yaxes(
    title=dict(text='Y', standoff=4),
    type='category',
    categoryorder='array',
    categoryarray=[f'{y:g}' for y in y_unique],
    range=monitor_y_range,
    domain=[0.35, 0.73],
    scaleanchor=monitor_xref,
    scaleratio=len(x_unique) / (len(y_unique) * (16 / 9)),
    constrain='domain',
    showticklabels=False,
    ticks='',
    showgrid=False,
    zeroline=False,
    fixedrange=True,
    row=1, col=2
)

fig_spatial.add_annotation(
    x=0.5,
    y=1.15,
    xref=f'{monitor_xref} domain',
    yref=f'{monitor_yref} domain',
    xanchor='center',
    yanchor='bottom',
    text='<b>Location on<br>the monitor</b>',
    showarrow=False,
    font=dict(size=12)
)

# ============================================================
# AXES AND DISPLAY
# ============================================================

fig_spatial.update_layout(
    title='2. Spatial mapping — voltage and detected events',
    autosize=True,
    height=500,
    template='plotly_white',
    dragmode='zoom',
    hovermode='closest',
    showlegend=False,
    margin=dict(l=65, r=15, t=60, b=50),
    font=dict(size=12)
)

fig_spatial.update_xaxes(
    showticklabels=False,
    row=1, col=1
)

fig_spatial.update_xaxes(
    title_text='Time (s)',
    row=2, col=1
)

fig_spatial.update_yaxes(
    title_text='Voltage (µV)',
    row=1, col=1
)

fig_spatial.update_yaxes(
    title_text='Events',
    range=[-1, 1],
    showticklabels=False,
    showgrid=False,
    zeroline=False,
    fixedrange=True,
    row=2, col=1
)

fig_spatial.show(config={
    'displaylogo': False,
    'scrollZoom': True,
    'responsive': True
})

print(f'Stimulus locations: {len(locations)}')
print(f'Presentations:      {len(spatial_trials)}')
print(f'Detected events:    {len(spike_times):,}')

## 3. Compare responses across screen locations

Each small panel represents one stimulus location on the monitor.
Within each panel, rows show repeated trials and black ticks mark detected events.

All panels use the same time axis. Dotted lines mark stimulus onset and offset.

**Question:** Which part of the monitor produces the strongest response?

In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

pre_stimulus_s = 0.2
post_stimulus_s = 0.2

x_positions = np.sort(np.unique([
    float(s['parameters']['x']) for s in spatial_trials
]))

y_positions = np.sort(np.unique([
    float(s['parameters']['y']) for s in spatial_trials
]))

if not monitor_y_increases_downward:
    y_positions = y_positions[::-1]

n_rows = len(y_positions)
n_cols = len(x_positions)

durations = np.array([
    s['offset_s'] - s['onset_s'] for s in spatial_trials
])

x_min = -pre_stimulus_s
x_max = durations.max() + post_stimulus_s
recording_end = len(voltage_filtered) / fs

location_trials = {
    (x, y): [
        s for s in spatial_trials
        if float(s['parameters']['x']) == x
        and float(s['parameters']['y']) == y
    ]
    for y in y_positions
    for x in x_positions
}

max_repetitions = max(
    len(trials) for trials in location_trials.values()
)

# A compact 16:9 plotting area, excluding margins
figure_width = 850
margins = dict(l=45, r=15, t=90, b=45)
figure_height = round(
    (figure_width - margins['l'] - margins['r']) * 9 / 16
    + margins['t'] + margins['b']
)

fig_grid_raster = make_subplots(
    rows=n_rows,
    cols=n_cols,
    shared_xaxes='all',
    shared_yaxes='all',
    horizontal_spacing=0.008 if n_cols > 1 else 0,
    vertical_spacing=0.012 if n_rows > 1 else 0
)

shapes = []

for ri, y in enumerate(y_positions, start=1):
    for ci, x in enumerate(x_positions, start=1):
        trials = location_trials[(x, y)]

        subplot = fig_grid_raster.get_subplot(ri, ci)
        xref = subplot.xaxis.plotly_name.replace('axis', '')
        yref = subplot.yaxis.plotly_name.replace('axis', '')

        shapes.append(dict(
            type='rect',
            xref=f'{xref} domain',
            yref=f'{yref} domain',
            x0=0, x1=1,
            y0=0, y1=1,
            fillcolor=location_colors.get((x, y), '#eeeeee'),
            opacity=0.12,
            line=dict(width=0),
            layer='below'
        ))

        if not trials:
            continue

        # Draw each spike as a thin vertical line in trial coordinates
        spike_x = []
        spike_y = []

        for repetition, s in enumerate(trials):
            onset = s['onset_s']

            start = np.searchsorted(
                spike_times,
                max(0, onset - pre_stimulus_s)
            )
            stop = np.searchsorted(
                spike_times,
                min(recording_end, s['offset_s'] + post_stimulus_s),
                side='left'
            )

            relative_times = spike_times[start:stop] - onset

            for spike_time in relative_times:
                spike_x.extend([spike_time, spike_time, None])
                spike_y.extend([
                    repetition - 0.32,
                    repetition + 0.32,
                    None
                ])

        fig_grid_raster.add_trace(go.Scattergl(
            x=spike_x,
            y=spike_y,
            mode='lines',
            line=dict(color='black', width=0.6),
            showlegend=False,
            hoverinfo='skip'
        ), row=ri, col=ci)

        # Stimulus onset
        shapes.append(dict(
            type='line',
            xref=xref,
            yref=f'{yref} domain',
            x0=0, x1=0,
            y0=0, y1=1,
            line=dict(
                color='rgba(40,40,40,0.35)',
                width=0.8,
                dash='dot'
            )
        ))

        trial_durations = np.array([
            s['offset_s'] - s['onset_s'] for s in trials
        ])

        if np.allclose(
            trial_durations, trial_durations[0],
            rtol=0, atol=1 / fs
        ):
            shapes.append(dict(
                type='line',
                xref=xref,
                yref=f'{yref} domain',
                x0=trial_durations[0],
                x1=trial_durations[0],
                y0=0, y1=1,
                line=dict(
                    color='rgba(40,40,40,0.35)',
                    width=0.8,
                    dash='dot'
                )
            ))
        else:
            for repetition, duration in enumerate(trial_durations):
                shapes.append(dict(
                    type='line',
                    xref=xref,
                    yref=yref,
                    x0=duration, x1=duration,
                    y0=repetition - 0.4,
                    y1=repetition + 0.4,
                    line=dict(
                        color='rgba(40,40,40,0.35)',
                        width=0.8,
                        dash='dot'
                    )
                ))

fig_grid_raster.update_xaxes(
    range=[x_min, x_max],
    showticklabels=False,
    ticks='',
    showgrid=False,
    zeroline=False,
    fixedrange=True
)

fig_grid_raster.update_yaxes(
    range=[max_repetitions - 0.5, -0.5],
    showticklabels=False,
    ticks='',
    showgrid=False,
    zeroline=False,
    fixedrange=True
)

# Overall spatial labels
fig_grid_raster.add_annotation(
    x=0.5, y=1.025,
    xref='paper', yref='paper',
    text='<b>X</b>',
    showarrow=False,
    xanchor='center',
    yanchor='bottom',
    font=dict(size=14)
)

fig_grid_raster.add_annotation(
    x=-0.025, y=0.5,
    xref='paper', yref='paper',
    text='<b>Y</b>',
    showarrow=False,
    xanchor='right',
    yanchor='middle',
    font=dict(size=14)
)

fig_grid_raster.add_annotation(
    x=0.5, y=-0.04,
    xref='paper', yref='paper',
    text='Within each panel: time → · repetitions stacked vertically',
    showarrow=False,
    xanchor='center',
    yanchor='top',
    font=dict(size=11, color='#555555')
)

fig_grid_raster.update_layout(
    title=dict(
        text='3. Trial-aligned rasters across the monitor locations',
        y=0.98,
        yanchor='top'
    ),
    shapes=shapes,
    width=figure_width,
    height=figure_height,
    margin=margins,
    template='plotly_white',
    plot_bgcolor='white',
    showlegend=False
)

fig_grid_raster.show(config={
    'displaylogo': False,
    'responsive': False,
    'staticPlot': True
})

### Before you continue

The rasters show individual events across trials, but comparing responses
across the whole monitor takes some effort.

**What type of plot would make it easier to see where on the monitor the
neuron responds most strongly?**

What would the axes show, and how would you represent the strength of the response?

## 4. Map the spatial response

Left: mean number of detected events during the stimulus, averaged across
repetitions at each tested location.

Right: the same data, linearly interpolated and Gaussian-smoothed.
Both maps use the same colour scale.

**Question:** Where is the strongest response? Does the smooth map contain
additional measurements, or only a different representation of the same data?

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import RegularGridInterpolator
from scipy.ndimage import gaussian_filter

# Smoothing width in units of the original grid spacing
smoothing_sigma = 0.6
fine_nx = 320
fine_ny = 180

# ============================================================
# COUNT EVENTS DURING EACH COMPLETE STIMULUS PRESENTATION
# ============================================================

x_unique = np.sort(np.unique([
    float(s['parameters']['x']) for s in spatial_trials
]))
y_unique = np.sort(np.unique([
    float(s['parameters']['y']) for s in spatial_trials
]))

event_sums = np.zeros((len(y_unique), len(x_unique)))
n_reps = np.zeros_like(event_sums, dtype=int)

recording_end = len(voltage_filtered) / fs

for s in spatial_trials:
    onset = s['onset_s']
    offset = s['offset_s']

    if onset < 0 or offset > recording_end or offset <= onset:
        continue

    xi = np.searchsorted(x_unique, float(s['parameters']['x']))
    yi = np.searchsorted(y_unique, float(s['parameters']['y']))

    # Count events in [onset, offset)
    start = np.searchsorted(spike_times, onset, side='left')
    stop = np.searchsorted(spike_times, offset, side='left')

    event_sums[yi, xi] += stop - start
    n_reps[yi, xi] += 1

rf_raw = np.full_like(event_sums, np.nan)
np.divide(event_sums, n_reps, out=rf_raw, where=n_reps > 0)

if not np.any(n_reps):
    raise ValueError('No complete stimulus presentations were found.')

if len(x_unique) < 2 or len(y_unique) < 2:
    raise ValueError('A 2D map requires at least two X and two Y positions.')

# ============================================================
# GRID EDGES: EACH BOX IS CENTRED ON ITS TESTED LOCATION
# ============================================================

x_edges = np.r_[
    x_unique[0] - (x_unique[1] - x_unique[0]) / 2,
    (x_unique[:-1] + x_unique[1:]) / 2,
    x_unique[-1] + (x_unique[-1] - x_unique[-2]) / 2
]

y_edges = np.r_[
    y_unique[0] - (y_unique[1] - y_unique[0]) / 2,
    (y_unique[:-1] + y_unique[1:]) / 2,
    y_unique[-1] + (y_unique[-1] - y_unique[-2]) / 2
]

# ============================================================
# LINEAR INTERPOLATION FOLLOWED BY GAUSSIAN SMOOTHING
# No cubic interpolation: avoids overshoot and ringing.
# ============================================================

# Fine-grid pixel centres
x_fine_edges = np.linspace(x_edges[0], x_edges[-1], fine_nx + 1)
y_fine_edges = np.linspace(y_edges[0], y_edges[-1], fine_ny + 1)

x_fine = (x_fine_edges[:-1] + x_fine_edges[1:]) / 2
y_fine = (y_fine_edges[:-1] + y_fine_edges[1:]) / 2

Xfine, Yfine = np.meshgrid(x_fine, y_fine)

# Hold the outermost measured values constant out to their box edges
query = np.column_stack([
    np.clip(Yfine.ravel(), y_unique.min(), y_unique.max()),
    np.clip(Xfine.ravel(), x_unique.min(), x_unique.max())
])

valid = np.isfinite(rf_raw)

value_interpolator = RegularGridInterpolator(
    (y_unique, x_unique),
    np.where(valid, rf_raw, 0.0),
    method='linear'
)

weight_interpolator = RegularGridInterpolator(
    (y_unique, x_unique),
    valid.astype(float),
    method='linear'
)

fine_values = value_interpolator(query).reshape(fine_ny, fine_nx)
fine_weights = weight_interpolator(query).reshape(fine_ny, fine_nx)

# Convert smoothing width from original-grid units to fine-grid pixels
sigma_x = (
    smoothing_sigma * np.median(np.diff(x_unique))
    / (x_fine_edges[1] - x_fine_edges[0])
)
sigma_y = (
    smoothing_sigma * np.median(np.diff(y_unique))
    / (y_fine_edges[1] - y_fine_edges[0])
)

smooth_values = gaussian_filter(
    fine_values, sigma=(sigma_y, sigma_x), mode='nearest'
)
smooth_weights = gaussian_filter(
    fine_weights, sigma=(sigma_y, sigma_x), mode='nearest'
)

rf_smooth = np.full_like(smooth_values, np.nan)
np.divide(
    smooth_values,
    smooth_weights,
    out=rf_smooth,
    where=smooth_weights > 1e-6
)

# ============================================================
# MATCHED COLOUR SCALE AND 16:9 MONITOR SHAPE
# ============================================================

vmax = max(float(np.nanmax(rf_raw)), 1.0)

cmap = plt.colormaps['viridis'].copy()
cmap.set_bad('#eeeeee')

fig, axes = plt.subplots(
    1, 2,
    figsize=(12, 4.5),
    constrained_layout=True
)

axes[0].pcolormesh(
    x_edges,
    y_edges,
    np.ma.masked_invalid(rf_raw),
    cmap=cmap,
    vmin=0,
    vmax=vmax,
    shading='flat',
    edgecolors='white',
    linewidth=0.4,
    antialiased=False
)

im = axes[1].imshow(
    rf_smooth,
    origin='lower',
    extent=[x_edges[0], x_edges[-1], y_edges[0], y_edges[-1]],
    cmap=cmap,
    vmin=0,
    vmax=vmax,
    interpolation='nearest',
    aspect='auto'
)

axes[0].set_title('Measured responses')
axes[1].set_title('Interpolated + Gaussian-smoothed')

for ax in axes:
    ax.set_xlim(x_edges[0], x_edges[-1])
    ax.set_ylim(y_edges[0], y_edges[-1])

    if monitor_y_increases_downward:
        ax.invert_yaxis()

    ax.set_aspect('auto')
    ax.set_box_aspect(9 / 16)
    ax.set_xlabel('Horizontal position (X)')
    ax.set_ylabel('Vertical position (Y)')

fig.colorbar(
    im,
    ax=axes.tolist(),
    shrink=0.7,
    pad=0.025,
    label='Mean detected events per stimulus'
)

fig.suptitle('4. Spatial response map', fontsize=14)
plt.show()